# ¿Falló el término, o le faltó coeficiente? — la medición

Este cuaderno **corre** el diagnóstico y nada más: no arma ninguna tabla ni conclusión — eso vive en `Benchmark_Noise_Diagnostic_Report_v1.ipynb`, que lee el `diagnostic.json` que esto deja y lo presenta.

Los techos de la campaña se buscaron con el material **limpio** y se mantienen fijos en los cinco niveles de contaminación. Esa decisión es deliberada y tiene un costo declarado: **una caída no se puede atribuir.** El término fallando bajo ruido y el coeficiente quedándose corto producen exactamente la misma curva. Esto es lo barato que separa las dos.

> **Tres puntos, y se paga uno solo.**
>
> Los dos brazos a este nivel bajo el techo limpio ya salen del barrido — están medidos y no se vuelven a correr. Lo que cuesta es la búsqueda del techo *sobre material contaminado*. Si el techo re-buscado recupera lo perdido, fue el coeficiente; si no lo recupera, fue el término.
>
> **`D` y `G`, y nadie más.** Son los dos métodos completos, uno por familia, y los únicos que llevan el coeficiente. `A` y `B` no tienen término de adaptación al que re-buscarle un techo, y `C`, `E` y `F` son ablaciones que multiplicarían la búsqueda sin agregar diagnóstico.
>
> **Sobre la transferencia de la curva y ninguna otra.** Buscar sobre las seis costaría seis veces lo que el diagnóstico vale y mediría cinco transferencias que la curva nunca recorrió, así que no habría contra qué leerlas.
>
> **El nivel es el tope del rango, fijado antes de que la curva existiera.** En el extremo el coeficiente está bajo la máxima presión, así que un techo re-buscado que no recupera nada ahí no recupera nada en ningún lado — y la lectura no depende de dónde alguien eligió mirar.
>
> **Sus números son de diagnóstico y no entran en las tablas del veredicto:** lo único que deciden es si vale reestructurar para techos por nivel. La re-búsqueda que se paga acá no gobierna ningún registro — `governs_the_ceilings_record` es falso bajo contaminación y sobre una transferencia sola.

In [ ]:
# Bootstrap: locate the repository wherever this is running, and import from it.
# Local, Colab and Kaggle differ only in where the checkout sits.
import os
import sys
from pathlib import Path


def find_repository() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for base in (os.environ.get("MIL_CREDA_REPO", ""), "/content", "/kaggle/working"):
        if base and Path(base).is_dir():
            candidates.append(Path(base))
            candidates.extend(sorted(Path(base).glob("*")))
    for candidate in candidates:
        if (candidate / "src" / "MIL_CREDA_Benchmark").is_dir():
            return candidate.resolve()
    raise SystemExit(
        "cannot find the repository. Set MIL_CREDA_REPO to the checkout that "
        "holds src/MIL_CREDA_Benchmark, or run this notebook from inside it."
    )


REPOSITORY = find_repository()
sys.path.insert(0, str(REPOSITORY / "src"))
print("repository:", REPOSITORY)

In [ ]:
import json

from MIL_CREDA_Benchmark import config, contamination, harness

tasa = config.NOISE_DIAGNOSTIC_LEVEL
device = harness.resolve_device()
# La escala configurada decide si esto es un ensayo, y el ensayo decide DÓNDE
# escribe --- la misma lectura y la misma razón que en la campaña y en el
# barrido. Escrito bajo `Results/Noise/` a secas, un diagnóstico de ensayo
# pisaría el de la corrida completa con números que no se pueden citar; el paso
# que corre este cuaderno se niega cuando la escala es la completa, porque sus
# `produces` nombran el árbol de ENSAYO y ninguno más.
ES_ENSAYO = config.is_pilot_scale()
reduccion = harness.Reduction(device=str(device), environment=harness.environment(),
                              labelNoise=tasa, pilot=ES_ENSAYO)

print(f"nivel de diagnóstico: ρ={tasa:g} "
      f"(el tope de {[f'{r:g}' for r in config.NOISE_LEVELS]})")
print("transferencia: {}->{}".format(*config.NOISE_TRANSFER))
print(f"brazos: {[config.NAME_OF[a] for a in config.NOISE_DIAGNOSTIC_ARMS]}")
print("escala:", "ensayo" if ES_ENSAYO else "completa",
      "| escribe en", config.noise_axis_for(reduccion.pilot))

## La medición

El techo buscado **sobre material contaminado**: es el punto que la campaña no
tiene y la única razón por la que este cuaderno cuesta algo. Una sola búsqueda, y
ninguna campaña — los otros dos puntos ya están en el registro del barrido.

In [ ]:
buscado = harness.search_ceilings(reduccion, device, noise=tasa,
                                  transfers=[config.NOISE_TRANSFER],
                                  pilot=reduccion.pilot)

## El registro

Lo que este cuaderno midió, escrito donde su paso declara que vive. El otro
extremo se lee y no se vuelve a correr: si esta celda y el barrido dijeran cosas
distintas habría dos versiones del mismo número.

In [ ]:
registro = {
    "level": tasa,
    "transfer": "{}->{}".format(*config.NOISE_TRANSFER),
    "arms": list(config.NOISE_DIAGNOSTIC_ARMS),
    "searchedUnderNoise": buscado,
    # Del BARRIDO y no de una campaña: a este nivel no hay campaña completa ---
    # sólo 0.0 y NOISE_REPORTED la tienen --- y la comparación es sobre la
    # transferencia del barrido de todos modos.
    "cleanCeilingRun": (contamination.load(tasa, kind="curve") or {}).get("summary"),
    "revision": config.REVISION,
    "diagnosticOnly": ("estos números no entran en las tablas del veredicto; "
                       "deciden si vale reestructurar para techos por nivel"),
}
# Bajo la raíz de ESTA corrida, compuesta desde la reducción con la que se
# midió y no desde una constante.
destino = config.noise_axis_for(reduccion.pilot)
destino.mkdir(parents=True, exist_ok=True)
(destino / "diagnostic.json").write_text(
    json.dumps(registro, indent=2, default=str), encoding="utf-8")

print("escrito:", destino / "diagnostic.json")
print()
print("la tabla y la conclusión las arma "
      "Benchmark_Noise_Diagnostic_Report_v1.ipynb")